# 

In [1]:
import pandas as pd
import regex

In [2]:
# NOTE: Licht & Sczepanski's coding scheme covered several types of groups; we only keep tpes "social group" (SG) and "implicit social group reference" (ISG)
id2label = {
    0: "O",
    6: 'B-SG', 1: 'I-SG', # social group reference
    10: 'B-SG', 5: 'I-SG', # implicit social group reference
}
id2label.update({i+5: 'B-other' for i in range(2, 5)})
id2label.update({i: 'I-other' for i in range(2, 5)})
encode_labels =  lambda labels: list(map(lambda l: id2label[l], labels))

In [3]:
def prepare_tokens(tokens, contractions = ['s', 't', 'm', 'll', 've', 'd', 're']):
    toks = []
    for i in range(0, len(tokens)-1):
        tok = tokens[i]
        # don't append whitespace if
        # (a) 
        #   - Pi	Initial Punctuation
        #   - Ps	Open Punctuation
        # (b) followed by 
        #   - Pc	Connector Punctuation
        #   - Pe	Close Punctuation
        #   - Pf	Final Punctuation
        #   - Po	Other Punctuation
        #   -       "'", "’" or "՛"
        # (c) digit with separator followed by digit
        # (d) is contraction like ('s 't)
        if not (
            # ... token is initial, opening, or connector punctuation or apostroph-like
            (regex.search(r"^[\p{Pi}\p{Ps}\p{Pc}]$", tokens[i]))
            or 
            # ... following token is connector, closing, final, or other punctuation
            regex.search(r'^[\p{Pc}\p{Pe}\p{Pf}\p{Po}]$', tokens[i+1])
            or 
            # ... token is digit preceeded by digits with decimal symbol
            (i>0 and regex.search(r'^\d{3}$', tok) and regex.search(r'\d[.,]$', tokens[i-1]))
            or 
            (i>0 and regex.search(r'^[,.]$', tokens[i]) and regex.search(r'\d$', tokens[i-1]) and regex.search(r'^\d{1,3}$', tokens[i+1]))
            or
            # ... token is apostroph-like and followed by contraction subwords (e.g. "'" in ["can", "'", "t"])
            (regex.search("['’՛]$", tok) and tokens[i+1].strip() in contractions)
            or 
            # ... following token is apostroph-like and the subsequent token is a contraction words (e.g. "can" in ["can", "'", "t"])
            (i<len(tokens)-1 and (regex.search("['’՛]$", tokens[i+1]) and tokens[i+2] in contractions))
        ):
            tok += ' '
        toks.append(tok)
    if tokens[-1]!=' ':
        toks.append(tokens[-1])
    
    return toks

# tests = [
#     ['Test', '(', 'with', 'parentheses', ')', '.'],
#     ['Test', 'with', '€', '100', ',', '000', '.', '00', '.'],
#     ['Cant', "'", 't', 'test', 'this', '.'],
#     ['Labour', "'", 's', "'", 'Proposal', "'", '.'],
#     ['Comma', ',', "comma"]
# ]
# for toks in tests:
#     print(res:=prepare_tokens(toks))
#     print(''.join(res))
#     print()

In [4]:
def convert_labeled_tokens_to_labeled_sequence(tokens, labels):
    entities = []
    pos = 0
    prev_k, prev_t = '', ''
    for i, (tok, lab) in enumerate(zip(tokens, labels)):
        if lab=='O':
            if len(entities)>0 and prev_t!='' and entities[-1][1] is None:
                entities[-1][1] = pos
            pos += len(tok)
            prev_k, prev_t = 'O', ''
            continue
        k, t = lab.split('-')
        # I should not begin span
        if k=='I' and prev_k=='O':
            k = 'B'
        # B should not be followed by B of same type
        if k=='B' and prev_k=='B' and prev_t==t:
            k = 'I'
        if k=='B':
            if len(entities)>0 and prev_t!=t and entities[-1][1] is None:
                entities[-1][1] = pos
            entities.append([pos, None, t])
            pos += len(tok)
        else:
            pos += len(tok)
            entities[-1][1] = pos-1 if tok[-1]==' ' else pos
        prev_k, prev_t = k, t

    if len(entities)>0 and entities[-1][1] is None:
        entities[-1][1] = pos

    text = ''.join(tokens)
    # cleanupt trailing non-word characters from spans
    for i in range(len(entities)):
        e = entities[i][1]
        if not regex.search(r'\w', text[e-1:e]):
            entities[i][1] -= 1
        
    return text, entities

def parse(doc):
    tokens = prepare_tokens(doc['tokens'])
    text, entities = convert_labeled_tokens_to_labeled_sequence(tokens=tokens, labels=encode_labels(doc['labels']['BSCModel']))
    out = {'id': doc['id'], 'text': text, 'label': entities}
    out['metadata'] = doc['metadata']
    return(out)

In [5]:
files = [
    "licht_detecting_2025-uk_manifestos.jsonl",
    "licht_detecting_2025-uk_parlspeech.jsonl",
    "licht_detecting_2025-de_manifestos.jsonl",
]

for file in files:
    output_file = file.replace('.jsonl', '-cleaned.jsonl')

    # read data
    data = pd.read_json(file, orient='records', lines=True)

    # parse records
    out = pd.DataFrame([parse(doc) for doc in data.to_dict(orient='records')])

    # write
    out.to_json(output_file, orient='records', lines=True, index=False, force_ascii=False)